In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.fact_lap_times"



In [0]:
# 1. Read raw lap_times table
df_lap_times_raw = spark.table(f"{catalog_name}.bronze.lap_times")



In [0]:
# 2. Transform & Cleanse
df_lap_times_silver = (
    df_lap_times_raw
    .replace(r"\N", None)
    .select(
        F.col("raceId").cast("integer").alias("race_id"),
        F.col("driverId").cast("integer").alias("driver_id"),
        F.col("lap").cast("integer").alias("lap_number"),
        F.col("position").cast("integer").alias("running_position"),
        F.col("time").alias("lap_time_formatted"),
        F.col("milliseconds").cast("long").alias("lap_time_ms")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_lap_times_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'fact',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")